In [1]:
from constants import DATA_ROOT_PATH_NAME, BANDPASS, HAMPEL_WINDOW_SIZE, HAMPEL_N_SIGMA, CROP_TMIN, CROP_TMAX, LOCAL_DETREND_WINDOW_SEC, LOCAL_DETREND_STEP_SEC, ASR_CUTOFF, ASR_BLOCKSIZE, ASR_WIN_LEN, ASR_WIN_OVERLAP, ASR_MAX_DROPOUT_FRACTION, ASR_MIN_CLEAN_FRACTION, ASR_MAX_BAD_CHANS

from preprocessing.step.bandpass import BandpassFilterStep
from preprocessing.step.detrend import LocalDetrendStep
from preprocessing.step.hampel import HampelFilterStep
from preprocessing.step.asr import ASRStep
from preprocessing.step.crop import CropStep

from preprocessing.pipeline import PreprocessingPipeline

from features.factory import CompleteFeatureExtractionEngine, FeatureExtractionConfig, CompleteFeatureExtractionResult
from features.categories import FeatureCategory
from features.dataset import SingleParticipantProcessedFeatureDatasetFactory, SingleParticipantProcessedFeatureDataset, FeaturesDataset
from features.io import FeaturesDatasetIO, SingleParticipantProcessedFeatureDatasetIO

from eeg.data import EEGRecordedDataProvider, EEGRecordedDataHelper

from features.name import FeatureNameHelper



%load_ext autoreload
%autoreload 2

# Création des 2 datasets de features

In [2]:
recordings = EEGRecordedDataProvider.build(DATA_ROOT_PATH_NAME, load_data=False)

/mnt/ssd2/pth-eeg/eeg/eeg/data.py:292: RuntimeWarning: Did not find any events.tsv associated with sub-001_task-eyesclosed.

The search_str was "data/sub-001/**/eeg/sub-001*events.tsv"
  raw_preview: mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:292: RuntimeWarning: Unable to map the following column(s) to to MNE:
Gender: F
Age: 57
Group: A
MMSE: 16
  raw_preview: mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:292: RuntimeWarning: Did not find any events.tsv associated with sub-002_task-eyesclosed.

The search_str was "data/sub-002/**/eeg/sub-002*events.tsv"
  raw_preview: mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:292: RuntimeWarning: Unable to map the following column(s) to to MNE:
Gender: F
Age: 78
Group: A
MMSE: 22
  raw_preview: mne.io.Raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:292:

In [3]:
tagged_recordings = EEGRecordedDataHelper.tag(recordings)

In [4]:
asr_pipeline = PreprocessingPipeline(name="ASR",
                                        steps=[
                                                BandpassFilterStep(BANDPASS),
                                                ASRStep(cutoff=ASR_CUTOFF, blocksize=ASR_BLOCKSIZE, win_len=ASR_WIN_LEN, win_overlap=ASR_WIN_OVERLAP, max_dropout_fraction=ASR_MAX_DROPOUT_FRACTION, min_clean_fraction=ASR_MIN_CLEAN_FRACTION, max_bad_chans=ASR_MAX_BAD_CHANS)
                                                ])

dethamp_pipeline = PreprocessingPipeline(name="det-hamp",
                                        steps=[ 
                                                BandpassFilterStep(BANDPASS),
                                                LocalDetrendStep(window_sec=LOCAL_DETREND_WINDOW_SEC, step_sec=LOCAL_DETREND_STEP_SEC),
                                                HampelFilterStep(window_size=HAMPEL_WINDOW_SIZE, n_sigma=HAMPEL_N_SIGMA)
                                                ])

In [5]:
categories_to_extract = [FeatureCategory.WAVELET, FeatureCategory.TEMPORAL, FeatureCategory.POWER_RATIO, FeatureCategory.SPECTRAL]
config = FeatureExtractionConfig(categories_to_extract=categories_to_extract)
feature_extraction_engine = CompleteFeatureExtractionEngine(config=config)

In [15]:
extraction_results= []
for eeg in tagged_recordings:
    # calibration une seule fois sur l'enregistrement source
    dethamp_pipeline.prepare(eeg)
    asr_pipeline.prepare(eeg)

    for window in EEGRecordedDataHelper.iter_split(eeg, t_start=10, window_seconds=60):
        dethamp_processed = dethamp_pipeline.compute(window, unload_source=True, prepare_steps=False)
        dethamp_extraction_result = feature_extraction_engine.extract(dethamp_processed)
        detamp_single_participant_dataset = SingleParticipantProcessedFeatureDatasetFactory.build(dethamp_extraction_result)
        SingleParticipantProcessedFeatureDatasetIO.export(detamp_single_participant_dataset, "computed_data/dethamp")
        dethamp_processed.unload()

        asr_processed = asr_pipeline.compute(window, unload_source=True, prepare_steps=False)
        asr_extraction_result = feature_extraction_engine.extract(asr_processed)
        asr_single_participant_dataset = SingleParticipantProcessedFeatureDatasetFactory.build(asr_extraction_result)
        SingleParticipantProcessedFeatureDatasetIO.export(asr_single_participant_dataset, "computed_data/asr")
        asr_processed.unload()

    dethamp_pipeline.clear_caches()
    asr_pipeline.clear_caches()

/mnt/ssd2/pth-eeg/eeg/eeg/data.py:275: RuntimeWarning: Did not find any events.tsv associated with sub-005_task-eyesclosed.

The search_str was "data/sub-005/**/eeg/sub-005*events.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:275: RuntimeWarning: Unable to map the following column(s) to to MNE:
Gender: M
Age: 70
Group: A
MMSE: 22
  raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:275: RuntimeWarning: Did not find any events.tsv associated with sub-005_task-eyesclosed.

The search_str was "data/sub-005/**/eeg/sub-005*events.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:275: RuntimeWarning: Unable to map the following column(s) to to MNE:
Gender: M
Age: 70
Group: A
MMSE: 22
  raw = read_raw_bids(bids_path=bids_path, verbose=False)
/mnt/ssd2/pth-eeg/eeg/eeg/data.py:275: RuntimeWarning: Did not find any events.tsv associated with sub-005_task-eyescl

# Import de toutes les features de tous les participants

In [2]:
dethamp_dataset = FeaturesDatasetIO.load("computed_data/dethamp")
asr_dataset = FeaturesDatasetIO.load("computed_data/asr")

# Tests statistiques

In [3]:
from stats.queries.factory import QueryFactoryConfig, QueryFactory, CorrectionSpec

factory = QueryFactory(
    QueryFactoryConfig.from_lists(
        subject_variables={"subject_age", "subject_mmse", "subject_id", "subject_health", "subject_gender"},
        ppc_bands=dethamp_dataset.ppc_band_names,
        psd_bands=dethamp_dataset.psd_band_names,
        eeg_features=dethamp_dataset.feature_names,
    )
)

fdr_correction = CorrectionSpec(
    method="fdr_bh",
    alpha=0.05,
)

In [31]:
from stats.runner import StatisticalTestRunner

queries = {feature_name : factory.compare(
    target=feature_name,
    group_a="Alzheimer",
    group_b="Healthy",
    correction=fdr_correction) for feature_name in dethamp_dataset.scalar_feature_names}


outcomes = StatisticalTestRunner.run(queries, asr_dataset)


# Entrainement d'un arbre de décision

In [3]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.tree import plot_tree, export_text
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError

import matplotlib.pyplot as plt

In [4]:
dethamp_dataset = dethamp_dataset.selector.filter_by_healthstate(["AD", "CN"])

In [28]:
from features.dataset import FeaturesDatasetSelector
from prediction.decision_tree.trainer import DecisionTreeTrainer
from prediction.decision_tree.tunning import HyperparameterSearch, HyperparameterGrid


TARGET = "subject_health"

METADATA_COLUMNS = [
    "subject_id",
    "subject_group",
    "subject_gender",
    "subject_age",
    "subject_mmse",
]

feature_name_factory = FeatureNameHelper(dethamp_dataset.all_feature_names)


eeg_feature_block = [
    "theta_alpha_ratio",
    "theta_beta_ratio",
    "gamma_alpha_ratio",
    "spectral_power_ratio",
    "spectral_centroid",
    "spectral_spread",
    "spectral_flux",
    "spectral_rolloff",
    "alpha_dominant_frequency",
    "gamma_dominant_frequency",
    "relative_wavelet_energy",
    "relative_wavelet_packet_energy",
]

selection = feature_name_factory.build(eeg=eeg_feature_block)
dataset = FeaturesDatasetSelector.select(dethamp_dataset, selection)

In [29]:
trainer = DecisionTreeTrainer(n_splits=5)
searcher = HyperparameterSearch(dataset=dataset, trainer=trainer, lambda_std=0.5)

hyperparameter_grid = HyperparameterGrid()
best_params, best_score = searcher.search(hyperparameter_grid)

🔍 Hyperparameter search (2 combinations)


100%|██████████| 2/2 [00:00<00:00,  3.06it/s]


✅ Best params found: DecisionTreeParameters(criterion='gini', max_depth=6, min_samples_split=4, min_samples_leaf=1, random_state=42)
Adjusted score = 0.7020


In [30]:
from prediction.decision_tree.feature_selection import FeatureSelector

feature_selector = FeatureSelector(trainer, feature_name_factory)
selected_feature_names, selected_columns, best_score = feature_selector.forward_selection(dataset, best_params)


🔍 Forward feature selection (max 20 blocks over 33 candidates)


Selecting blocks:   5%|▌         | 1/20 [00:03<01:02,  3.30s/it]

Added spectral_power_ratio (19 cols) -> score=0.6741 ± 0.0726


Selecting blocks:  10%|█         | 2/20 [00:07<01:07,  3.73s/it]

Added spectral_spread (19 cols) -> score=0.7065 ± 0.0844


...

Added alpha_dominant_frequency (19 cols) -> score=0.7287 ± 0.0710


Selecting blocks:  25%|██▌       | 5/20 [00:23<01:14,  4.98s/it]

Added skewness (19 cols) -> score=0.7388 ± 0.0690


Selecting blocks:  25%|██▌       | 5/20 [00:29<01:27,  5.83s/it]

Stopping: no further improvement.


In [33]:
selected_feature_names

['spectral_power_ratio',
 'spectral_spread',
 'gamma_alpha_ratio',
 'alpha_dominant_frequency',
 'skewness']